# Part 1: Loading, Inspection and Cleaning

**CS 22B, Fall 2026 · Team 12** · Section lead: Aleksandra (Ola) Szymula

This notebook loads the Allen Cell Types Database cell feature file, inspects it, and cleans it. It saves a cleaned file, `data/neurons_clean.csv`, that the analysis and visualization sections use.

**Data source:** Allen Institute for Brain Science, Allen Cell Types Database, https://celltypes.brain-map.org
**File:** https://celltypes.brain-map.org/cell_types_specimen_details.csv
**One row = one neuron** recorded in a slice of mouse or human brain tissue.

## 1. Setup

The next cell downloads the CSV from the Allen Institute into a `data` folder, unless it is already there. This works the same in Google Colab and in Jupyter, so no manual upload is needed.

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd

DATA_URL = "https://celltypes.brain-map.org/cell_types_specimen_details.csv"
DATA_PATH = "data/cell_types_specimen_details.csv"
CLEAN_PATH = "data/neurons_clean.csv"

os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

## 2. Load and inspect the raw data

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Rows, columns:", raw.shape)

In [ ]:
raw.columns.tolist()

In [ ]:
raw.dtypes.value_counts()

In [ ]:
raw.head()

In [ ]:
raw.tail()

## 3. Select and rename the columns we need

The raw file has many columns with long names (for example `ef__vrest`). We keep only the columns tied to our research questions and give them short, readable names. The `ef__` columns are electrophysiology features.

| Raw column | New name | Meaning |
| --- | --- | --- |
| `specimen__id` | `cell_id` | Unique ID of the neuron |
| `donor__species` | `species` | Human or mouse |
| `structure__acronym` | `brain_region` | Brain area the cell came from |
| `structure__layer` | `layer` | Cortical layer |
| `tag__dendrite_type` | `dendrite_type` | Spiny (mostly excitatory) or aspiny (mostly inhibitory) |
| `ef__vrest` | `resting_potential_mv` | Resting membrane potential (mV) |
| `ef__ri` | `input_resistance_mohm` | Input resistance (MΩ) |
| `ef__avg_firing_rate` | `firing_rate_hz` | Average firing rate (Hz) |
| `ef__threshold_i_long_square` | `rheobase_pa` | Smallest current that makes the cell fire (pA) |
| `ef__upstroke_downstroke_ratio_long_square` | `upstroke_downstroke_ratio` | Spike shape; a higher ratio means a narrower spike |
| `ef__tau` | `membrane_tau_ms` | Membrane time constant (ms) |

The next cell checks that every one of these columns exists before we use it.

In [ ]:
column_names = {
    "specimen__id": "cell_id",
    "donor__species": "species",
    "structure__acronym": "brain_region",
    "structure__layer": "layer",
    "tag__dendrite_type": "dendrite_type",
    "ef__vrest": "resting_potential_mv",
    "ef__ri": "input_resistance_mohm",
    "ef__avg_firing_rate": "firing_rate_hz",
    "ef__threshold_i_long_square": "rheobase_pa",
    "ef__upstroke_downstroke_ratio_long_square": "upstroke_downstroke_ratio",
    "ef__tau": "membrane_tau_ms",
}

missing_columns = [col for col in column_names if col not in raw.columns]
print("Missing columns:", missing_columns)

In [ ]:
neurons = raw[list(column_names)].rename(columns=column_names)
neurons.head()

In [ ]:
numeric_columns = [
    "resting_potential_mv",
    "input_resistance_mohm",
    "firing_rate_hz",
    "rheobase_pa",
    "upstroke_downstroke_ratio",
    "membrane_tau_ms",
]
category_columns = ["species", "brain_region", "layer", "dendrite_type"]

neurons.info()

In [ ]:
neurons[numeric_columns].describe().round(2)

## 4. Clean category labels

Category labels need to be consistent before we group by them. We check every value in each category column, then:

- convert species names to `human` and `mouse`,
- make dendrite types lowercase and remove extra spaces,
- store layers as text, because labels like `2/3` and `6a` are not numbers.

In [ ]:
for col in ["species", "dendrite_type", "layer"]:
    print(neurons[col].value_counts(dropna=False), "\n")

In [ ]:
species_names = {"Homo Sapiens": "human", "Mus musculus": "mouse"}
neurons["species"] = neurons["species"].str.strip().replace(species_names)

neurons["dendrite_type"] = neurons["dendrite_type"].str.strip().str.lower()
neurons["layer"] = neurons["layer"].astype("string").str.strip()

for col in ["species", "dendrite_type", "layer"]:
    print(neurons[col].value_counts(dropna=False), "\n")

### Layer labels differ by species

The cortex has six layers, numbered from the surface down. The table below counts cells per layer label for each species. Mouse and human cells use different labels for the same layers (for example, `2/3` versus separate `2` and `3`). If we compared layers without fixing this, some layer groups would contain only one species, and a layer comparison would really be a species comparison.

In [ ]:
pd.crosstab(neurons["layer"], neurons["species"])

We merge the labels into one shared set: layers 2 and 3 become `2/3`, and layers 6a and 6b become `6`. The original label is kept in a new column, `layer_original`, so the change can be checked.

In [ ]:
neurons["layer_original"] = neurons["layer"]

layer_merge = {"2": "2/3", "3": "2/3", "6a": "6", "6b": "6"}
neurons["layer"] = neurons["layer"].replace(layer_merge)

pd.crosstab(neurons["layer"], neurons["species"])

## 5. Duplicates

Each neuron should appear once. We check for fully repeated rows and for repeated cell IDs.

In [ ]:
print("Fully duplicated rows:", neurons.duplicated().sum())
print("Repeated cell IDs:", neurons["cell_id"].duplicated().sum())

In [ ]:
rows_before = len(neurons)
neurons = neurons.drop_duplicates(subset="cell_id")
duplicates_removed = rows_before - len(neurons)
print("Duplicate rows removed:", duplicates_removed)

## 6. Missing values

We look at the share of missing values in each column. A cell without electrophysiology measurements cannot help answer any of our questions, so we remove cells where **all** numeric features are missing. A cell missing only some values (for example, its layer) stays in the data; each later analysis uses the cells that have the values it needs.

In [ ]:
missing_share = (neurons.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_share.to_frame("percent missing")

In [ ]:
rows_before = len(neurons)
neurons = neurons.dropna(subset=numeric_columns, how="all")
no_ephys_removed = rows_before - len(neurons)
print("Cells removed with no measurements:", no_ephys_removed)

## 7. Invalid values

Some values are physically impossible or far outside what neurons show. We use these limits:

| Column | Valid range | Reason |
| --- | --- | --- |
| `resting_potential_mv` | −100 to −40 mV | Healthy neurons rest at a negative voltage, usually about −80 to −60 mV |
| `input_resistance_mohm` | above 0 | Resistance cannot be zero or negative |
| `firing_rate_hz` | 0 or more | A firing rate cannot be negative |
| `rheobase_pa` | above 0 | A cell needs positive current to fire |
| `membrane_tau_ms` | above 0 | A time constant must be positive |

We replace values outside these limits with missing values (`NaN`) instead of deleting the whole row, so the cell's other valid measurements are kept.

In [ ]:
valid = {
    "resting_potential_mv": neurons["resting_potential_mv"].between(-100, -40),
    "input_resistance_mohm": neurons["input_resistance_mohm"] > 0,
    "firing_rate_hz": neurons["firing_rate_hz"] >= 0,
    "rheobase_pa": neurons["rheobase_pa"] > 0,
    "membrane_tau_ms": neurons["membrane_tau_ms"] > 0,
}

invalid_counts = {}
for col, is_valid in valid.items():
    is_invalid = neurons[col].notna() & ~is_valid
    invalid_counts[col] = int(is_invalid.sum())
    neurons.loc[is_invalid, col] = np.nan

pd.Series(invalid_counts, name="values set to NaN").to_frame()

## 8. Summary of the cleaned data

In [ ]:
cleaning_summary = pd.DataFrame({
    "step": [
        "Raw rows",
        "Duplicate cells removed",
        "Cells with no measurements removed",
        "Invalid values set to NaN",
        "Rows after cleaning",
    ],
    "count": [
        len(raw),
        duplicates_removed,
        no_ephys_removed,
        sum(invalid_counts.values()),
        len(neurons),
    ],
})
cleaning_summary

In [ ]:
neurons.groupby(["species", "dendrite_type"]).size().unstack(fill_value=0)

The table above shows how many cells each species and dendrite type group has. Groups with only a few cells will give less reliable averages, so the analysis section should check these counts before comparing groups.

Only 13 human cells are sparsely spiny, which is too few to compare. We keep these cells in the cleaned data, and the analysis section leaves them out of spiny vs. aspiny comparisons.

In [ ]:
neurons.to_csv(CLEAN_PATH, index=False)
print("Saved", neurons.shape, "to", CLEAN_PATH)